<h2>IS453 Financial Analytics</h2><h2>Group Assignment - Home Improvement Loan Scorecard</h2>

### Step 0: Customer Segmentation

This notebook selects the customer segment used for the Home Improvement Loan credit scorecard. It covers **segmentation only**: no bureau merge, EDA, feature engineering, WOE/IV, train-test split or modelling is performed here.

## 0.1 Business Context

Our objective is to develop a credit scorecard for **Home Improvement Loans** targeting **adult property owners with positive reported income**.

We adopt an **inclusive segmentation strategy** to preserve as many potentially relevant customers as possible, including self-employed individuals, retirees and homeowners with different housing arrangements.

Segmentation should not prematurely exclude higher-risk borrowers. Eligibility rules only define *who could plausibly apply* for the product; the scorecard will subsequently assess *creditworthiness*.

**Key assumption:** the dataset does not explicitly identify renovation-loan purposes. Therefore, the selected rows are a **proxy population of potential home improvement borrowers**, not a verified sample of renovation loans.

In [1]:
import numpy as np
import pandas as pd

# show all columns and format floats for readability
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 0.2 Load and Inspect Raw Application Data

Load the raw application data and keep an untouched copy (`app_data_orig`) so that every segmentation step can be compared back to the original population.

In [2]:
# load the raw application data
data_path = 'data/raw/IS453 Group Assignment - Application Data.csv'
app_data = pd.read_csv(data_path)

# make a copy of the original data for reference
app_data_orig = app_data.copy()

print('Original dataset shape:', app_data_orig.shape)
print('Unique SK_ID_CURR values:', f"{app_data_orig['SK_ID_CURR'].nunique():,}")

Original dataset shape: (307511, 120)
Unique SK_ID_CURR values: 307,511


**Distribution of TARGET (0 = good, 1 = bad)**

In [3]:
# count goods and bads, including any missing or non-binary values
target_dist = app_data_orig['TARGET'].value_counts(dropna=False).sort_index().to_frame('Count')
target_dist['Percent (%)'] = target_dist['Count'] / target_dist['Count'].sum() * 100
target_dist.index = target_dist.index.map({0: '0 - Good', 1: '1 - Bad'}).fillna('Other / missing')
target_dist

,Count,Percent (%)
TARGET,,
0 - Good,282686,91.9271
1 - Bad,24825,8.0729


**Unique values of FLAG_OWN_REALTY**

In [4]:
# check the homeownership flag values, including missing
app_data_orig['FLAG_OWN_REALTY'].value_counts(dropna=False).to_frame('Count')

,Count
FLAG_OWN_REALTY,
Y,213312
N,94199


**Missing and invalid values in the eligibility variables**

- `DAYS_BIRTH` is recorded as negative days relative to the application date, so a missing or non-negative value is invalid.
- `AMT_INCOME_TOTAL` must be a positive number, so a missing, zero or negative value is invalid.

In [5]:
# check missing and invalid values for the eligibility variables
eligibility_check = pd.DataFrame({
    'Missing values': [app_data_orig['DAYS_BIRTH'].isnull().sum(),
                       app_data_orig['AMT_INCOME_TOTAL'].isnull().sum()],
    'Invalid values': [(app_data_orig['DAYS_BIRTH'] >= 0).sum(),
                       (app_data_orig['AMT_INCOME_TOTAL'] <= 0).sum()],
    'Invalid rule': ['DAYS_BIRTH >= 0', 'AMT_INCOME_TOTAL <= 0']
}, index=['DAYS_BIRTH', 'AMT_INCOME_TOTAL'])

eligibility_check

,Missing values,Invalid values,Invalid rule
DAYS_BIRTH,0,0,DAYS_BIRTH >= 0
AMT_INCOME_TOTAL,0,0,AMT_INCOME_TOTAL <= 0


In [6]:
# review the range of age and income in the original data
age_years = -app_data_orig['DAYS_BIRTH'] / 365.25

print(f"Age range (years): {age_years.min():.1f} to {age_years.max():.1f}")
print(f"Income range: {app_data_orig['AMT_INCOME_TOTAL'].min():,.0f} to {app_data_orig['AMT_INCOME_TOTAL'].max():,.0f}")

Age range (years): 20.5 to 69.1
Income range: 25,650 to 117,000,000


The raw file has one row per unique `SK_ID_CURR`, a binary `TARGET`, and only `Y`/`N` values for `FLAG_OWN_REALTY`. No missing or invalid values were found in `DAYS_BIRTH` or `AMT_INCOME_TOTAL`, and every applicant is already 18 or older. The age and income filters below are still applied explicitly so that the eligibility rules are documented and would catch invalid rows in future data.

## 0.3 Define Segmentation Criteria

| Characteristic | Selection rule | Business justification |
|---|---|---|
| Homeownership (`FLAG_OWN_REALTY`) | `Y` | A property owner may undertake renovations, repairs and residential improvements. |
| Adult eligibility (`DAYS_BIRTH`) | Age ≥ 18 | Adult borrowers may qualify regardless of employment status or retirement age. |
| Positive income (`AMT_INCOME_TOTAL`) | Greater than 0 | Applicants must report positive income without imposing a minimum affordability threshold. |

**Characteristics we deliberately do NOT filter on**

- **`NAME_CONTRACT_TYPE`**: all historical contract types are retained for maximum inclusion, acknowledging that revolving credit behaves differently from instalment credit.
- **`NAME_INCOME_TYPE`**: all income categories (working, commercial associate, pensioner, state servant, etc.) are included provided income is positive.
- **`NAME_HOUSING_TYPE`**: owners do not necessarily live in the property they own, so a homeowner living with parents or renting elsewhere is still eligible.
- **Maximum age**: older homeowners may also need to finance repairs or accessibility improvements.
- **Credit history or delinquency**: these are predictors for the scorecard, not eligibility criteria.
- **`TARGET`**: both good and bad borrowers are retained; filtering on the outcome would bias the model.

## 0.4 Apply Segmentation Filters

The three filters are applied one at a time to new DataFrames, so the original data is never overwritten and the impact of each rule can be measured. Only rows are filtered; all 120 original columns are kept.

Rows with missing or invalid eligibility values cannot satisfy a rule, so they are excluded at that step and counted explicitly.

In [7]:
# Filter 1: homeowners only
homeowners = app_data_orig[app_data_orig['FLAG_OWN_REALTY'] == 'Y'].copy()

print('Rows with missing FLAG_OWN_REALTY:', app_data_orig['FLAG_OWN_REALTY'].isnull().sum())
print(f'Homeowners: {len(homeowners):,} rows')

Rows with missing FLAG_OWN_REALTY: 0
Homeowners: 213,312 rows


In [8]:
# Filter 2: adult homeowners, age calculated from DAYS_BIRTH
age_years = -homeowners['DAYS_BIRTH'] / 365.25

# missing or non-negative DAYS_BIRTH is invalid and cannot pass the age rule
invalid_age = homeowners['DAYS_BIRTH'].isnull() | (homeowners['DAYS_BIRTH'] >= 0)
print('Rows with missing/invalid DAYS_BIRTH:', invalid_age.sum())
print('Rows under 18:', ((age_years < 18) & ~invalid_age).sum())

adult_homeowners = homeowners[~invalid_age & (age_years >= 18)].copy()
print(f'Adult homeowners: {len(adult_homeowners):,} rows')

Rows with missing/invalid DAYS_BIRTH: 0
Rows under 18: 0
Adult homeowners: 213,312 rows


In [9]:
# Filter 3: adult homeowners with positive income
invalid_income = adult_homeowners['AMT_INCOME_TOTAL'].isnull()
print('Rows with missing AMT_INCOME_TOTAL:', invalid_income.sum())
print('Rows with income <= 0:', (adult_homeowners['AMT_INCOME_TOTAL'] <= 0).sum())

segment_df = adult_homeowners[adult_homeowners['AMT_INCOME_TOTAL'] > 0].copy()
print(f'Adult homeowners with positive income: {len(segment_df):,} rows')

# confirm all columns are preserved
print('Columns preserved:', segment_df.shape[1] == app_data_orig.shape[1])

Rows with missing AMT_INCOME_TOTAL: 0
Rows with income <= 0: 0
Adult homeowners with positive income: 213,312 rows
Columns preserved: True


## 0.5 Evaluate Segmentation Impact

For each stage we calculate the remaining rows, rows removed from the previous stage, retention versus the original data, and the number of goods and bads.

The bad rate uses only valid binary `TARGET` values (0 or 1) in the denominator.

In [10]:
# function to summarise goods, bads and bad rate for a dataframe
def target_summary(df):
    goods = (df['TARGET'] == 0).sum()
    bads = (df['TARGET'] == 1).sum()
    known = goods + bads
    bad_rate = bads / known * 100 if known > 0 else np.nan
    return goods, bads, bad_rate

stages = {
    'Original application dataset': app_data_orig,
    'Homeowners only': homeowners,
    'Adult homeowners': adult_homeowners,
    'Adult homeowners with positive income': segment_df
}

original_rows = len(app_data_orig)
previous_rows = original_rows
rows = []

for stage, df in stages.items():
    current_rows = len(df)
    goods, bads, bad_rate = target_summary(df)
    rows.append({
        'Stage': stage,
        'Resulting rows': current_rows,
        'Rows removed': previous_rows - current_rows,
        '% of original retained': current_rows / original_rows * 100,
        'Number of goods': goods,
        'Number of bads': bads,
        'Bad rate (%)': bad_rate
    })
    previous_rows = current_rows

stage_table = pd.DataFrame(rows).set_index('Stage')

stage_table.style.format({
    'Resulting rows': '{:,.0f}',
    'Rows removed': '{:,.0f}',
    '% of original retained': '{:.2f}%',
    'Number of goods': '{:,.0f}',
    'Number of bads': '{:,.0f}',
    'Bad rate (%)': '{:.2f}%'
})

,Resulting rows,Rows removed,% of original retained,Number of goods,Number of bads,Bad rate (%)
Stage,,,,,,
Original application dataset,"307,511",0,100.00%,"282,686","24,825",8.07%
Homeowners only,"213,312","94,199",69.37%,"196,329","16,983",7.96%
Adult homeowners,"213,312",0,69.37%,"196,329","16,983",7.96%
Adult homeowners with positive income,"213,312",0,69.37%,"196,329","16,983",7.96%


### Final Resulting Rows and Bad Rate

In [11]:
# compare the original dataset with the final segment
orig_goods, orig_bads, original_bad_rate = target_summary(app_data_orig)
final_goods, final_bads, final_bad_rate = target_summary(segment_df)
final_rows = len(segment_df)

final_summary = pd.DataFrame({
    'Original dataset': [f'{original_rows:,}', f'{orig_goods:,}', f'{orig_bads:,}',
                         f'{original_bad_rate:.2f}%', '100.00%'],
    'Final segment': [f'{final_rows:,}', f'{final_goods:,}', f'{final_bads:,}',
                      f'{final_bad_rate:.2f}%', f'{final_rows / original_rows * 100:.2f}%']
}, index=['Total rows', 'Number of good borrowers', 'Number of bad borrowers',
          'Bad rate (%)', 'Percentage of original rows retained'])
final_summary.index.name = 'Metric'

final_summary

,Original dataset,Final segment
Metric,,
Total rows,"307,511","213,312"
Number of good borrowers,"282,686","196,329"
Number of bad borrowers,"24,825","16,983"
Bad rate (%),8.07%,7.96%
Percentage of original rows retained,100.00%,69.37%


In [12]:
# calculate overall change from the original dataset to the final segment
rows_excluded = original_rows - final_rows
retention_pct = final_rows / original_rows * 100 if original_rows > 0 else np.nan
bad_rate_change_pp = final_bad_rate - original_bad_rate
bad_rate_change_rel = ((final_bad_rate / original_bad_rate) - 1) * 100 if original_bad_rate > 0 else np.nan

print(f'Total rows excluded:                  {rows_excluded:,}')
print(f'Overall retention:                    {retention_pct:.2f}%')
print(f'Change in bad rate:                   {bad_rate_change_pp:+.2f} percentage points')
print(f'Relative change in bad rate:          {bad_rate_change_rel:+.2f}%')

Total rows excluded:                  94,199
Overall retention:                    69.37%
Change in bad rate:                   -0.11 percentage points
Relative change in bad rate:          -1.38%


**Interpretation**

The final segment has a **slightly lower observed bad rate (7.96%) than the original dataset (8.07%)**, a change of about −0.11 percentage points (around −1.4% in relative terms). Homeowners are marginally lower risk than the overall population, which is consistent with property ownership indicating some financial stability. The difference is small, so the segment's risk profile stays close to the full portfolio.

All 94,199 excluded rows come from the homeownership filter. The age and income filters removed no rows, because every applicant in this data is already an adult (minimum age about 20.5 years) with positive reported income.

## 0.6 Business Interpretation

1. **Borrowers remaining:** 213,312 applicants remain after segmentation.
2. **Retention:** about 69.4% of the original 307,511 applications are retained.
3. **Goods and bads:** the segment contains 196,329 good and 16,983 bad observations. That is a large number of bads, enough to support stable WOE binning and logistic regression later.
4. **Bad rate comparison:** the segment bad rate (7.96%) is slightly below the original (8.07%), so the segment is neither cherry-picked low-risk nor concentrated in high-risk borrowers.
5. **Inclusiveness:** the approach remains inclusive. Only one rule (homeownership) actually removes rows. All contract types, income types (including pensioners and commercial associates) and housing arrangements are kept, there is no maximum age, and no affordability threshold beyond positive income. Credit history is left for the scorecard to assess.
6. **Limitations of the homeownership proxy:**
   - The data does not record loan purpose, so these are past loans of homeowners, **not verified renovation loans**. Repayment behaviour on other products may differ from home improvement lending.
   - `FLAG_OWN_REALTY` is self-reported and gives no detail on property value, equity, mortgage status or ownership share, all of which drive renovation demand and capacity.
   - Some renovation borrowers may be excluded, for example tenants with landlord permission or people about to complete a property purchase.
   - The segment mixes cash and revolving loans, which have different repayment structures from a typical instalment home improvement loan.

## 0.7 Export Filtered Dataset

Export the segment with all original application columns so it can be used for bureau data aggregation and merging in the next step. Check that each `SK_ID_CURR` appears only once before exporting.

In [13]:
# check for duplicated applicant IDs before export
duplicate_ids = segment_df['SK_ID_CURR'].duplicated().sum()
print('Duplicated SK_ID_CURR values:', duplicate_ids)
assert duplicate_ids == 0, 'Duplicate applicant IDs found'

# confirm key columns and all original columns are present
assert {'SK_ID_CURR', 'TARGET'}.issubset(segment_df.columns)
assert list(segment_df.columns) == list(app_data_orig.columns)

# export the segment
output_path = 'data/home_improvement_segment.csv'
segment_df.to_csv(output_path, index=False)

Duplicated SK_ID_CURR values: 0


In [14]:
import os

# display final dataset details
print('Final dataset shape:', segment_df.shape)
print('Unique applicant IDs:', f"{segment_df['SK_ID_CURR'].nunique():,}")
print('Export location:', os.path.abspath(output_path))

Final dataset shape: (213312, 120)
Unique applicant IDs: 213,312
Export location: c:\Users\leeli\OneDrive\Desktop\Fa\data\home_improvement_segment.csv
